In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset, Dataset, WeightedRandomSampler
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from util import filter_data, seed_everything
from util import mask_crop as mask_crop_fn
from validate import val_model, ValLoaderWrapper
from loader import QSM_c1_Dataset as QSM_RAM_Dataset
from networks import QSMDecoder
from datetime import datetime
import math

# ============================================================
# GATED CLINICAL-VISUAL WRAPPER
# ============================================================
class GatedResNetWrapper(nn.Module):
    def __init__(self, base_model, clinical_dim, feat_dim=512):
        super().__init__()
        self.base_model = nn.Sequential(*list(base_model.children())[:-1])
        
        self.clinical_gate = nn.Sequential(
            nn.Linear(clinical_dim, 128),
            nn.ReLU(),
            nn.Linear(128, feat_dim),
            nn.Sigmoid()
        )
        
        self.fusion = nn.Sequential(
            nn.Linear(feat_dim + clinical_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, img, clin=None, clinical_vec=None):
        c = clin if clin is not None else clinical_vec
        if c is None:
            raise ValueError("Clinical data must be provided as 'clin' or 'clinical_vec'")

        vis_feats_raw = self.base_model(img) 
        vis_feats = vis_feats_raw.view(img.size(0), -1)
        
        gate = self.clinical_gate(c)
        gated_vis = vis_feats * gate
        
        combined = torch.cat([gated_vis, c], dim=1)
        logits = self.fusion(combined)
        
        return logits, vis_feats_raw

# ============================================================
# LORA IMPLEMENTATION
# ============================================================
class LoRALayer(nn.Module):
    def __init__(self, original_layer, rank=4, alpha=8):
        super().__init__()
        self.original_layer = original_layer
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        dev = original_layer.weight.device
        
        for param in self.original_layer.parameters():
            param.requires_grad = False
            
        if isinstance(original_layer, nn.Conv2d):
            out_channels, in_channels, k_h, k_w = original_layer.weight.shape
            self.lora_A = nn.Parameter(torch.zeros(rank, in_channels, k_h, k_w, device=dev))
            self.lora_B = nn.Parameter(torch.zeros(out_channels, rank, 1, 1, device=dev))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B) 
        elif isinstance(original_layer, nn.Linear):
            out_f, in_f = original_layer.weight.shape
            self.lora_A = nn.Parameter(torch.zeros(rank, in_f, device=dev))
            self.lora_B = nn.Parameter(torch.zeros(out_f, rank, device=dev))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)

    def forward(self, x):
        orig_out = self.original_layer(x)
        if isinstance(self.original_layer, nn.Conv2d):
            lora_out = torch.nn.functional.conv2d(x, self.lora_A, 
                                                 padding=self.original_layer.padding, 
                                                 stride=self.original_layer.stride)
            lora_out = torch.nn.functional.conv2d(lora_out, self.lora_B)
        else:
            lora_out = (x @ self.lora_A.t()) @ self.lora_B.t()
        return orig_out + lora_out * self.scaling

# ============================================================
# EXPERIMENT CONFIG
# ============================================================
EXP_NAME = "gated_lora_autoencoder" 
device = 'cuda:0'
LIMIT_SUBS = None 
CACHE_PATH = 'qsm_preprocessed_cache.pt'
LOAD_FROM_CACHE = True 
seed_everything(0)

# ============================================================
# DATA PREPARATION
# ============================================================
nii_path = '/data2/ali/dbs/qsm/'
seg_path = '/data2/ali/dbs/seg_ps/'
file_dir = '/data2/ali/dbs/dbs_03292024.csv'
cv_features = {'Age', 'Sex', 'Ethnicity', 'Race', 'Disease Duration (year)', ' pre op levadopa equivalent dose (mg)', ' Test medication status', ' OFF (pre-dbs updrs)', ' ON (pre-dbs updrs)'}
all_needed_cols = cv_features | {'CORNELL ID', ' OFF meds ON stim 6mo'}
motor_df = filter_data(file_dir, all_needed_cols, True)

for col in [' OFF (pre-dbs updrs)', ' ON (pre-dbs updrs)', ' OFF meds ON stim 6mo']:
    motor_df[col] = pd.to_numeric(motor_df[col], errors='coerce')

motor_df = motor_df.dropna(subset=[' OFF (pre-dbs updrs)', ' OFF meds ON stim 6mo'])
improvement_ratios = (motor_df[' OFF (pre-dbs updrs)'] - motor_df[' OFF meds ON stim 6mo']) / motor_df[' OFF (pre-dbs updrs)']
label_map = {str(int(row['CORNELL ID'])): (1 if ratio >= 0.30 else 0) for (_, row), ratio in zip(motor_df.iterrows(), improvement_ratios)}

cols_to_norm = ['Age', 'Disease Duration (year)', ' OFF (pre-dbs updrs)', ' pre op levadopa equivalent dose (mg)']
for col in cols_to_norm:
    motor_df[col] = pd.to_numeric(motor_df[col], errors='coerce')
    col_mean, col_std = motor_df[col].mean(), motor_df[col].std()
    motor_df[col] = (motor_df[col] - col_mean) / (col_std + 1e-8)

clinical_dict = {str(int(row['CORNELL ID'])): row[list(cv_features)].values.astype(np.float32) for _, row in motor_df.iterrows()}
full_dataset = QSM_RAM_Dataset(nii_path, seg_path, mask_crop_fn, clinical_dict, label_map, limit=LIMIT_SUBS, cache_path=CACHE_PATH, load_cache=LOAD_FROM_CACHE, return_index=True)
actual_clin_dim = next(iter(clinical_dict.values())).shape[0]
full_dataset.clin_dim = actual_clin_dim

all_cached_ids = {str(k) for k in full_dataset.volumes.keys()}
label_keys = {str(k) for k in label_map.keys()}
unique_labeled_subs = np.array(sorted(list(all_cached_ids & label_keys)))
unique_sub_labels = np.array([label_map[sid] for sid in unique_labeled_subs])
unlabeled_ids = np.array(list(all_cached_ids - label_keys))

class RandomMasking(object):
    def __init__(self, mask_size=8, num_masks=2, p=0.5):
        self.mask_size, self.num_masks, self.p = mask_size, num_masks, p
    def __call__(self, tensor):
        if torch.rand(1).item() > self.p: return tensor
        _, h, w = tensor.shape
        for _ in range(self.num_masks):
            y, x = torch.randint(0, h - self.mask_size, (1,)), torch.randint(0, w - self.mask_size, (1,))
            tensor[:, y:y+self.mask_size, x:x+self.mask_size] = 0
        return tensor

qsm_aug = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    RandomMasking(p=0.3)
])

# ============================================================
# CROSS-VALIDATION LOOP
# ============================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"gated_ae_log_{timestamp}.txt"
def log_print(message):
    print(message)
    with open(log_filename, "a") as f: f.write(message + "\n")

all_split_best_metrics = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

for split, (t_p_idx, v_p_idx) in enumerate(skf.split(unique_labeled_subs, unique_sub_labels)):
    train_subs, val_subs = unique_labeled_subs[t_p_idx], unique_labeled_subs[v_p_idx]
    t_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(train_subs)]
    v_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(val_subs)]
    pt_subs = np.concatenate([unlabeled_ids, train_subs])
    pt_idx = [i for i, s in enumerate(full_dataset.samples) if str(s['sub_id']) in set(pt_subs)]

    train_labels = [label_map[str(full_dataset.samples[i]['sub_id'])] for i in t_idx]
    class_counts = np.bincount(train_labels)
    sampler = WeightedRandomSampler([1./class_counts[l] for l in train_labels], 2*len(t_idx))

    pt_loader = DataLoader(Subset(full_dataset, pt_idx), batch_size=48, shuffle=True)
    t_loader = DataLoader(Subset(full_dataset, t_idx), batch_size=48, sampler=sampler)
    v_loader = DataLoader(Subset(full_dataset, v_idx), batch_size=48, shuffle=False)

    base_resnet = models.resnet18(weights='IMAGENET1K_V1')
    base_resnet.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    model = GatedResNetWrapper(base_resnet, clinical_dim=actual_clin_dim).to(device)
    decoder = QSMDecoder(feat_dim=512).to(device)

    # --- A. AUTOENCODER PRETRAINING (No Diffusion) ---
    optimizer_pt = torch.optim.Adam(list(model.parameters()) + list(decoder.parameters()), lr=1e-4)
    criterion_pt = nn.MSELoss()
    full_dataset.train_mode, full_dataset.transform = True, qsm_aug
    
    for pt_epoch in range(10):
        model.train(); decoder.train()
        for imgs, clin, _, _ in pt_loader:
            imgs, clin = imgs.to(device), clin.to(device)
            optimizer_pt.zero_grad()
            _, feats = model(imgs, clin)
            recon = decoder(feats)
            criterion_pt(recon, imgs).backward(); optimizer_pt.step()

    # --- B. FINE-TUNING (LoRA Injection) ---
    for param in model.base_model.parameters(): param.requires_grad = False
    
    for layer_idx in [6, 7]: # Sequential indices for layer3 and layer4
        target_layer = model.base_model[layer_idx]
        for block in target_layer:
            if hasattr(block, "conv1"): block.conv1 = LoRALayer(block.conv1, rank=16)
            if hasattr(block, "conv2"): block.conv2 = LoRALayer(block.conv2, rank=16)
            if block.downsample is not None and isinstance(block.downsample[0], nn.Conv2d):
                block.downsample[0] = LoRALayer(block.downsample[0], rank=16)

    optimizer = torch.optim.Adam([
        {'params': [p for n, p in model.named_parameters() if "lora_" in n], 'lr': 1e-4},
        {'params': model.clinical_gate.parameters(), 'lr': 1e-4},
        {'params': model.fusion.parameters(), 'lr': 5e-5}
    ], weight_decay=1e-3)
    
    MAX_EPOCHS = 100 
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-7)
    loss_fn = nn.CrossEntropyLoss().to(device)
    
    best_score, best_metrics_this_split, patience = 0, None, 0
    os.makedirs(f"weights/{EXP_NAME}", exist_ok=True)

    for epoch in range(MAX_EPOCHS):
        model.train(); full_dataset.train_mode = True
        for imgs, clin, lbls, _ in t_loader:
            imgs, clin, lbls = imgs.to(device), clin.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits, _ = model(imgs, clin)
            loss = loss_fn(logits, lbls)
            loss.backward(); optimizer.step()
        
        scheduler.step()
        model.eval(); full_dataset.train_mode, full_dataset.transform = False, None
        m = val_model(ValLoaderWrapper(v_loader), device, model, loss_fn, v_loader.dataset)
        
        current_score = (m[3] * m[4] * m[5]) ** (1/3)
        log_print(f"S{split} E{epoch} | Acc: {m[1]:.4f} | Sens: {m[3]:.4f} | Spec: {m[4]:.4f} | AUC: {m[5]:.4f}")

        if current_score > best_score:
            best_score, best_metrics_this_split, patience = current_score, m, 0
            torch.save(model.state_dict(), f"weights/{EXP_NAME}/split_{split}.pth")
        else:
            patience += 1
        if patience >= 30: break
    
    if best_metrics_this_split is not None: all_split_best_metrics.append(best_metrics_this_split)

# ============================================================
# FINAL SUMMARY
# ============================================================
if len(all_split_best_metrics) > 0:
    final_metrics = np.array(all_split_best_metrics)
    avg_metrics, std_metrics = np.mean(final_metrics, axis=0), np.std(final_metrics, axis=0)
    print("\n" + "="*45 + "\nGATED-LORA + AUTOENCODER RESULTS\n" + "="*45)
    names = ["Loss", "Accuracy", "Precision", "Sensitivity", "Specificity", "AUC"]
    for i, name in enumerate(names):
        print(f"{name:<15} : {avg_metrics[i]:.4f} ± {std_metrics[i]:.4f}")
    print("="*45)
else:
    print("No splits completed successfully.")

Keeping CORNELL ID
Keeping Age
Keeping Sex
Keeping Ethnicity
Keeping Race
Keeping Disease Duration (year)
Keeping  OFF (pre-dbs updrs)
Keeping  ON (pre-dbs updrs)
Keeping  pre op levadopa equivalent dose (mg)
Keeping  Test medication status
Keeping  OFF meds ON stim 6mo
Loaded cache with 7776 slices from 108 subjects
S0 E0 | Acc: 0.7232 | Sens: 0.5174 | Spec: 0.8056 | AUC: 0.6253
S0 E1 | Acc: 0.6726 | Sens: 0.5382 | Spec: 0.7264 | AUC: 0.6200
S0 E2 | Acc: 0.6518 | Sens: 0.5451 | Spec: 0.6944 | AUC: 0.6146
S0 E3 | Acc: 0.6468 | Sens: 0.5347 | Spec: 0.6917 | AUC: 0.6278
S0 E4 | Acc: 0.6488 | Sens: 0.4444 | Spec: 0.7306 | AUC: 0.5965
S0 E5 | Acc: 0.6468 | Sens: 0.4722 | Spec: 0.7167 | AUC: 0.6029
S0 E6 | Acc: 0.5982 | Sens: 0.5208 | Spec: 0.6292 | AUC: 0.6047
S0 E7 | Acc: 0.6081 | Sens: 0.5104 | Spec: 0.6472 | AUC: 0.6148
S0 E8 | Acc: 0.5942 | Sens: 0.4861 | Spec: 0.6375 | AUC: 0.6059
S0 E9 | Acc: 0.6032 | Sens: 0.5000 | Spec: 0.6444 | AUC: 0.6093
S0 E10 | Acc: 0.5992 | Sens: 0.4861 | Spe